In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Microsoft YaHei'  

from dask_jobqueue import SLURMCluster
from dask.distributed import Client
from glob import glob
import pandas as pd
import dask
import dask.dataframe as dd
from dask import delayed
import re
import pandas as pd
import jieba

In [ ]:
cluster = SLURMCluster(queue='caslake', cores=1, memory='160GB', 
                       processes=1, walltime='10:00:00', interface='ib0',
                       job_extra_directives=['--account=macs40123']
                      )
cluster.scale(jobs=1)

In [ ]:
! squeue -u yjia2

In [ ]:
client = Client(cluster)
client

# Data  Clean

In [ ]:
xlsx_files = sorted(glob("data/*.xlsx"))
assert xlsx_files, "No .xlsx files found under data/"

In [ ]:
def load_excel_fixed(path: str) -> pd.DataFrame:
    df = pd.read_excel(path, sheet_name=0, engine="openpyxl", dtype=str)
    expected_cols = ["??", "??", "????", "??", "????"]
    missing = [c for c in expected_cols if c not in df.columns]
    if missing:
        raise ValueError(f"{path} : {missing}")
    df = df[expected_cols]
    df["??"] = pd.to_numeric(df["??"], errors="raise").astype("Int64")
    for c in ["??","????","??","????"]:
        df[c] = df[c].astype(str)
    df["??"] = pd.to_numeric(df["??"], errors="coerce")
    df = df.dropna(subset=expected_cols)
    return df

In [ ]:
delayed_parts = [delayed(load_excel_fixed)(p) for p in xlsx_files]

meta = pd.DataFrame({
    "??": pd.Series(dtype="Int64"),
    "??": pd.Series(dtype="object"),
    "????": pd.Series(dtype="object"),
    "??": pd.Series(dtype="object"),
    "????": pd.Series(dtype="object"),
})

ddf = dd.from_delayed(delayed_parts, meta=meta)

out_dir = "merged_parquet"
ddf.to_parquet(
    out_dir,
    engine="pyarrow",
    write_index=False,
    partition_on=["??"]
)

print("? Merged and partitioned Parquet written to:", out_dir)

In [ ]:
ddf_check = dd.read_parquet("merged_parquet")
print(ddf_check.npartitions)  
print(ddf_check.head())


In [ ]:
MARK_PATTERNS = [
    r"????? ? ? ????",
    r"??:???????????",
    r"(????,???????)",
    r"????",
    r"???????",
    r"?????????????",
    r"????????????",
    r"????:?????????",
    r"(?? ?????)",
    r"?????"
]

RE_PUNCT = re.compile(r"[,?!??;:????()?????????????,.!?;:\"'()\-]+")
RE_BRACKET = re.compile(r"?[^?]*?")

with open("baidu_stopwords.txt", "r", encoding="utf-8") as f:
    STOPWORDS = set([line.strip() for line in f if line.strip()])

def clean_and_segment(text: str, year_val) -> str:
    if not isinstance(text, str):
        text = "" if pd.isna(text) else str(text)

    text = RE_BRACKET.sub("", text)

    try:
        y = int(year_val)
    except Exception:
        y = None
    if y is not None and y <= 2002:
        text = re.sub(r"^.*???[::]", "", text)

    for pat in MARK_PATTERNS:
        text = re.sub(pat, "", text)

    text = RE_PUNCT.sub(" ", text)   

    seg_list = jieba.cut(text)
    seg_text = " ".join(seg_list)

    # Delete all non-Chinese characters (including punctuation, numbers, 
    # and English letters)
    seg_text = re.sub(r"[^ \u4e00-\u9fff]", "", seg_text)

    seg_text = re.sub(r"\s+", " ", seg_text).strip()

    tokens = [w for w in seg_text.split() if w not in STOPWORDS]
    seg_text = " ".join(tokens)

    return seg_text


In [ ]:
import dask.dataframe as dd

ddf = dd.read_parquet("merged_parquet")

# Perform word segmentation cleaning in parallel on each partition
ddf = ddf.assign(
    ????_?? = ddf.map_partitions(
        lambda df: df.apply(
            lambda row: clean_and_segment(row["????"], row["??"]),
            axis=1
        ),
        meta=("????_??", "object")
    )
)

# Write out new Parquet (keeping year partitions)
out_dir = "cleaned_segmented_parquet"
ddf.to_parquet(
    out_dir,
    engine="pyarrow",
    write_index=False,
    partition_on=["??"]
)
print("? Cleaned and segmented text written to:", out_dir)


In [ ]:
ddf_cleaned = dd.read_parquet("cleaned_segmented_parquet")

print(ddf_cleaned.head(5))


# EDA

In [ ]:
ddf_cleaned = dd.read_parquet("cleaned_segmented_parquet")

In [ ]:
ddf_cleaned = ddf_cleaned.categorize(columns=["??"])
ddf_cleaned = ddf_cleaned.assign(??=ddf_cleaned["??"].astype("int64"))

ddf_cleaned = ddf_cleaned.rename(columns={
    "??": "date",
    "????": "edition",
    "??": "title",
    "????": "content_raw",
    "????_??": "content_cleaned",
    "??": "year"
})

print("Total rows:", len(ddf_cleaned))
print("Columns:", list(ddf_cleaned.columns))
print(
    "Year range:",
    ddf_cleaned["year"].min().compute(),
    "-",
    ddf_cleaned["year"].max().compute()
)


In [ ]:
print("\nData types:")
print(ddf_cleaned.dtypes)

In [ ]:
articles_per_year = ddf_cleaned.groupby("year").size().compute()
print("\nArticles per year:")
print(articles_per_year.head())

# Visualize
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'DejaVu Serif']
articles_per_year.plot(kind="line", figsize=(10,4), title=
                       "Number of Articles per Year")
plt.xlabel("Year")
plt.ylabel("Article Count")
plt.show()

In [ ]:
ddf_cleaned["content_length"] = ddf_cleaned["content_cleaned"].str.len()
ddf_cleaned = ddf_cleaned.persist()
avg_length = (
    ddf_cleaned.groupby("year")["content_length"]
    .mean()
    .compute()
    .sort_index()
)
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'DejaVu Serif']
plt.rcParams['axes.unicode_minus'] = False

# Visualize
plt.figure(figsize=(10,5))
plt.plot(avg_length.index, avg_length.values, marker='o', linewidth=1.8)
plt.title("Average Cleaned Article Length per Year", fontsize=14)
plt.xlabel("Year", fontsize=12)
plt.ylabel("Average Article Length (characters)", fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Filter articles containing "masses"
mask = ddf_cleaned["content_cleaned"].str.contains("??", na=False)

# Count the number of articles that appear each year
crowd_per_year = (
    ddf_cleaned[mask]
    .groupby("year")
    .size()
    .compute()
    .sort_index()
)

total_crowd_articles = ddf_cleaned[ddf_cleaned["content_cleaned"].str.contains
                                   ("??", na=False)].shape[0].compute()
print("Total number of articles containing 'People':", total_crowd_articles)

# Visualize
plt.rcParams['font.family'] = 'serif'
plt.rcParams['font.serif'] = ['Times New Roman', 'Times', 'DejaVu Serif']
plt.rcParams['axes.unicode_minus'] = False

plt.figure(figsize=(10, 5))
plt.plot(crowd_per_year.index, crowd_per_year.values, marker="o", linewidth=1.8, 
         color="steelblue")
plt.title("Number of Articles Containing 'People' per Year", fontsize=14)
plt.xlabel("Year", fontsize=12)
plt.ylabel("Article Count", fontsize=12)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()
